# 🚀 High-Accuracy YOLO11m Fine-Tuning with Advanced Augmentations
**Google Colab T4 GPU Pipeline**

Key Improvements in this configuration:
- 🔍 **High-Resolution Input (`imgsz=1024`)**: Retains thin lattice tower beams and monopole shafts that were previously blurred at 640px.
- 🎨 **Copy-Paste (`copy_paste=0.3`)**: Multiplies rare monopole towers onto various background terrains.
- 🧩 **Mosaic + MixUp (`mosaic=1.0, mixup=0.15`)**: Teaches the model to ignore cluttered background foliage and power lines.
- 📐 **Scale Jittering (`scale=0.5`)**: Handles extreme drone zoom levels and flight altitudes.
- 📈 **Cosine LR & Regularization**: Prevents memorization and enforces real generalization on unseen scenes.

### Step 1: Check GPU & Install Ultralytics

In [ ]:
!nvidia-smi
!pip install ultralytics

### Step 2: Upload Dataset (`yolo_dataset.zip`)
Upload `yolo_dataset.zip` from your local folder `A:\Electrohack\` (if not already extracted).

In [ ]:
from google.colab import files
import zipfile, os

if not os.path.exists('/content/dataset'):
    if not os.path.exists('yolo_dataset.zip'):
        print('Please upload yolo_dataset.zip:')
        uploaded = files.upload()
    !mkdir -p /content/dataset
    with zipfile.ZipFile('/content/yolo_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')

# Ensure data.yaml is configured
yaml_content = """
path: /content/dataset
train: train/images
val: val/images
test: test/images
nc: 2
names:
  - supporting_tower
  - monopole_tower
"""
with open('/content/dataset/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print('Dataset ready!')

### Step 3: Run High-Accuracy Augmented Training on GPU (~5 to 7 minutes)

In [ ]:
from ultralytics import YOLO

# Load clean pretrained model
model = YOLO('yolo11m.pt')

# Train with high-resolution and advanced augmentations
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=120,
    imgsz=1024,         # Higher resolution (preserves thin structures)
    batch=8,            # Optimal for 1024px on 15GB T4 GPU
    patience=40,
    device=0,           # GPU
    seed=42,
    project='/content/runs/detect',
    name='yolo11m_augmented',
    save=True,
    plots=True,
    # --- Advanced Augmentations ---
    mosaic=1.0,         # 4-image mosaic
    mixup=0.15,         # Background blending
    copy_paste=0.30,    # Balances rare monopole towers
    scale=0.50,         # Zoom jitter (+/- 50%)
    degrees=10.0,       # Rotation (+/- 10 deg)
    fliplr=0.5,         # Horizontal flip
    hsv_h=0.015,        # Color/lighting variation
    hsv_s=0.7,
    hsv_v=0.4,
    cos_lr=True,        # Cosine learning rate
    weight_decay=0.001, # Anti-overfitting regularization
    close_mosaic=10     # Disable mosaic for last 10 epochs
)

print('High-Accuracy Training Complete!')

### Step 4: Evaluate Directly on the Test Set

In [ ]:
# Evaluate the new best.pt directly on the test split
best_model = YOLO('/content/runs/detect/yolo11m_augmented/weights/best.pt')
test_metrics = best_model.val(
    data='/content/dataset/data.yaml',
    split='test',
    imgsz=1024,
    device=0,
    project='/content/runs/detect',
    name='yolo11m_augmented_test',
    save_json=True,
    plots=True
)

print('Test evaluation complete!')

### Step 5: Download All Enhanced Results & Model

In [ ]:
import shutil
from google.colab import files

# Package the enhanced run
shutil.make_archive('/content/yolo11m_augmented_results', 'zip', '/content/runs/detect/yolo11m_augmented')
files.download('/content/yolo11m_augmented_results.zip')

# Package test evaluation
shutil.make_archive('/content/yolo11m_augmented_test_results', 'zip', '/content/runs/detect/yolo11m_augmented_test')
files.download('/content/yolo11m_augmented_test_results.zip')